# 02 — Multiclass Temporal Leakage AuditExtends the binary audit to the full multiclass label space, after excluding two preprocessing pitfalls documented in the paper (Section 4.2): categorical port features and ultra-rare classes (fewer than 10 real samples). Produces the results in Table 5, and the structural unseen-class finding (Section 6.2, 41.1% of Friday's traffic).**Note:** this is the final, corrected version of the script. During development we diagnosed two real bugs (unreliable SMOTE for ultra-rare classes, and a Destination Port / LightGBM leaf-wise interaction) before arriving at the version below — see the paper's Section 4.2 for the full account.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from imblearn.over_sampling import SMOTE
from collections import Counter
import gc, warnings
warnings.filterwarnings('ignore')

SAVE_PATH    = '/content/drive/MyDrive/New Approach/cicids2017-01/cicids2017/preprocessed'
LABEL_COL    = 'Label'
MIN_SAMPLES  = 6
RANDOM_STATE = 42
ordered_days = ['monday','tuesday','wednesday','thursday','friday']

ULTRA_RARE_EXCLUDE = ['Infiltration', 'Web Attack - Sql Injection']
# ── root cause fix: port numbers are categorical identifiers,
# not continuous quantities — standardizing them produces
# artificially zero-variance regions that LightGBM's leaf-wise
# splitting exploits pathologically (verified: 49.8% -> 99.77%
# accuracy on identical data after exclusion) ──
CATEGORICAL_EXCLUDE = ['Destination Port', 'Source Port']

if 'day_dfs' not in dir() or len(day_dfs) == 0:
    day_dfs = {}
    for day in ordered_days:
        path = SAVE_PATH + f'/{day}_preprocessed.csv'
        if not os.path.exists(path): continue
        df        = pd.read_csv(path, low_memory=False)
        df['day'] = day
        day_dfs[day] = df
        print(f'{day:12s}: {len(df):>8,} rows')
    print(f'Total: {sum(len(v) for v in day_dfs.values()):,}')


def preprocess_multiclass(df_tr, df_te, label_col=LABEL_COL,
                          variance_thresh=0.01, corr_thresh=0.95,
                          min_samples=MIN_SAMPLES):
    try:
        df_tr = df_tr[~df_tr[label_col].isin(ULTRA_RARE_EXCLUDE)].reset_index(drop=True)
        df_te = df_te[~df_te[label_col].isin(ULTRA_RARE_EXCLUDE)].reset_index(drop=True)

        X_tr     = df_tr.drop(columns=[label_col,'day'], errors='ignore').copy()
        X_te     = df_te.drop(columns=[label_col,'day'], errors='ignore').copy()
        y_tr_raw = df_tr[label_col].copy()
        y_te_raw = df_te[label_col].copy()

        # ── drop categorical port columns before any numeric processing ──
        port_cols_tr = [c for c in X_tr.columns if c in CATEGORICAL_EXCLUDE]
        port_cols_te = [c for c in X_te.columns if c in CATEGORICAL_EXCLUDE]
        X_tr.drop(columns=port_cols_tr, inplace=True, errors='ignore')
        X_te.drop(columns=port_cols_te, inplace=True, errors='ignore')

        X_tr.drop(columns=X_tr.select_dtypes(exclude=[np.number]).columns, inplace=True)
        X_te.drop(columns=X_te.select_dtypes(exclude=[np.number]).columns, inplace=True)
        X_tr.replace([np.inf,-np.inf], np.nan, inplace=True)
        X_te.replace([np.inf,-np.inf], np.nan, inplace=True)
        mask_tr  = X_tr.notna().all(axis=1)
        mask_te  = X_te.notna().all(axis=1)
        X_tr     = X_tr[mask_tr].reset_index(drop=True)
        y_tr_raw = y_tr_raw[mask_tr].reset_index(drop=True)
        X_te     = X_te[mask_te].reset_index(drop=True)
        y_te_raw = y_te_raw[mask_te].reset_index(drop=True)

        counts_tr = y_tr_raw.value_counts()
        valid_tr  = counts_tr[counts_tr >= min_samples].index
        keep_tr   = y_tr_raw.isin(valid_tr)
        X_tr      = X_tr[keep_tr].reset_index(drop=True)
        y_tr_raw  = y_tr_raw[keep_tr].reset_index(drop=True)

        common = [c for c in X_tr.columns if c in X_te.columns]
        X_tr = X_tr[common].copy(); X_te = X_te[common].copy()

        vt   = VarianceThreshold(threshold=variance_thresh)
        arr  = vt.fit_transform(X_tr)
        cols = np.array(common)[vt.get_support()]
        X_tr = pd.DataFrame(arr, columns=cols)
        X_te = pd.DataFrame(vt.transform(X_te), columns=cols)

        corr  = X_tr.corr().abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        drop  = [c for c in upper.columns if any(upper[c] >= corr_thresh)]
        X_tr.drop(columns=drop, inplace=True)
        X_te.drop(columns=drop, inplace=True, errors='ignore')
        feature_names = X_tr.columns.tolist()

        scaler  = StandardScaler()
        X_tr_sc = pd.DataFrame(scaler.fit_transform(X_tr), columns=feature_names)
        X_te_sc = pd.DataFrame(scaler.transform(X_te),     columns=feature_names)

        le = LabelEncoder()
        le.fit(sorted(y_tr_raw.unique()))
        train_classes = set(le.classes_)
        n_classes = len(train_classes)

        y_tr = pd.Series(le.transform(y_tr_raw), name='label')

        te_known_mask  = y_te_raw.isin(train_classes)
        y_te_known_raw = y_te_raw[te_known_mask].reset_index(drop=True)
        X_te_known     = X_te_sc[te_known_mask.values].reset_index(drop=True)
        y_te_known     = pd.Series(le.transform(y_te_known_raw), name='label')

        unseen_classes = sorted(set(y_te_raw.unique()) - train_classes)
        n_unseen_rows  = (~te_known_mask).sum()
        n_known_classes_in_test = y_te_known.nunique()

        print(f'  Features: {len(feature_names)} (port columns excluded) | Train classes: {n_classes}')
        print(f'  Train dist: {dict(Counter(y_tr_raw))}')
        print(f'  Unseen-in-test classes: {unseen_classes} ({n_unseen_rows:,} rows)')
        print(f'  Known classes actually present in test: {n_known_classes_in_test}')

        counts  = Counter(y_tr)
        maj_cls = max(counts, key=counts.get)
        maj_cnt = counts[maj_cls]
        min_cnt = min(counts.values())
        k       = max(1, min(5, min_cnt - 1))
        strat   = {c: max(v, min(20_000, maj_cnt))
                   for c, v in counts.items() if c != maj_cls}

        smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=k,
                      sampling_strategy=strat)
        X_sm, y_sm = smote.fit_resample(X_tr_sc, y_tr)
        print(f'  After SMOTE: {len(y_sm):,} | Test (known): {len(y_te_known):,} '
              f'| Test (unseen): {n_unseen_rows:,}')

        return (X_sm, y_sm, X_te_known, y_te_known, le,
                feature_names, unseen_classes, n_unseen_rows, len(y_te_raw),
                n_classes, n_known_classes_in_test)

    except Exception as e:
        import traceback
        print(f'  [ERROR]: {e}')
        traceback.print_exc()
        return (None,)*11


def evaluate_multiclass(model, X_te, y_te, le, model_name, split_name,
                        unseen_classes=None, n_unseen=0, n_total_test=0,
                        n_known_classes_in_test=0):
    if n_known_classes_in_test <= 1:
        print(f'  [{split_name:8s}] {model_name:14s} | '
              f'DEGENERATE: only {n_known_classes_in_test} known class in test '
              f'({n_unseen/n_total_test*100:.1f}% of traffic is unseen-class). '
              f'Known-class metrics not meaningful for this day.')
        return {
            'model': model_name, 'split': split_name,
            'accuracy_known': None, 'f1_macro_known': None,
            'f1_weighted_known': None, 'mean_class_recall_known': None,
            'unseen_class_fraction': round(n_unseen/n_total_test, 4) if n_total_test else None,
            'n_unseen_classes': len(unseen_classes) if unseen_classes else 0,
            'n_known_classes_in_test': n_known_classes_in_test,
            'degenerate': True,
        }

    preds = model.predict(X_te)
    acc = accuracy_score(y_te, preds)
    f1m = f1_score(y_te, preds, average='macro',    zero_division=0)
    f1w = f1_score(y_te, preds, average='weighted', zero_division=0)

    present_classes = sorted(y_te.unique())
    cm = confusion_matrix(y_te, preds, labels=present_classes)
    support = cm.sum(axis=1)
    diag    = cm.diagonal()
    valid   = support > 0
    per_class_recall  = np.where(valid, diag / np.where(support==0,1,support), np.nan)
    mean_known_recall = np.nanmean(per_class_recall)

    unseen_fraction = n_unseen / n_total_test if n_total_test > 0 else 0.0

    print(f'  [{split_name:8s}] {model_name:14s} | '
          f'Acc(known):{acc:.4f} | F1m(known):{f1m:.4f} | '
          f'MeanClassRecall:{mean_known_recall:.4f} | '
          f'UnseenClassFraction:{unseen_fraction:.4f} | '
          f'TestClassesPresent:{len(present_classes)}')

    return {
        'model': model_name, 'split': split_name,
        'accuracy_known': round(acc,4),
        'f1_macro_known': round(f1m,4),
        'f1_weighted_known': round(f1w,4),
        'mean_class_recall_known': round(float(mean_known_recall),4),
        'unseen_class_fraction': round(unseen_fraction,4),
        'n_unseen_classes': len(unseen_classes) if unseen_classes else 0,
        'n_known_classes_in_test': n_known_classes_in_test,
        'degenerate': False,
    }


# ── build TEMPORAL split ──
print('\nBuilding TEMPORAL multiclass split...')
df_tr_raw = pd.concat(
    [day_dfs[d] for d in ['monday','tuesday','wednesday','thursday']],
    ignore_index=True)
df_te_raw = day_dfs['friday'].copy()

if len(df_tr_raw) > 200_000:
    df_tr_raw = (df_tr_raw.groupby(LABEL_COL, group_keys=False)
                 .apply(lambda x: x.sample(
                     min(len(x), max(MIN_SAMPLES,
                         int(200_000*len(x)/len(df_tr_raw)))),
                     random_state=RANDOM_STATE))
                 .reset_index(drop=True))
if len(df_te_raw) > 100_000:
    df_te_raw = (df_te_raw.groupby(LABEL_COL, group_keys=False)
                 .apply(lambda x: x.sample(
                     min(len(x), max(MIN_SAMPLES,
                         int(100_000*len(x)/len(df_te_raw)))),
                     random_state=RANDOM_STATE))
                 .reset_index(drop=True))

result = preprocess_multiclass(df_tr_raw, df_te_raw)
assert result[0] is not None, "Temporal multiclass split failed"
(X_tr_temp_mc, y_tr_temp_mc, X_te_temp_mc, y_te_temp_mc,
 le_temp_mc, feats_temp_mc, unseen_temp, n_unseen_temp,
 n_total_te_temp, n_classes_temp, n_known_test_temp) = result
del df_tr_raw, df_te_raw; gc.collect()
print('Temporal multiclass split ready ✓')


# ── build RANDOM split ──
print('\nBuilding RANDOM multiclass split...')
df_all = pd.concat(day_dfs.values(), ignore_index=True)
df_all = df_all[~df_all[LABEL_COL].isin(ULTRA_RARE_EXCLUDE)].reset_index(drop=True)
counts_all = df_all[LABEL_COL].value_counts()
df_all = df_all[df_all[LABEL_COL].isin(
    counts_all[counts_all >= MIN_SAMPLES*2].index)].reset_index(drop=True)

if len(df_all) > 200_000:
    df_all = (df_all.groupby(LABEL_COL, group_keys=False)
              .apply(lambda x: x.sample(
                  min(len(x), max(MIN_SAMPLES,
                      int(200_000*len(x)/len(df_all)))),
                  random_state=RANDOM_STATE))
              .reset_index(drop=True))

X_all = df_all.drop(columns=[LABEL_COL,'day'], errors='ignore')
y_all = df_all[LABEL_COL]
X_r_tr, X_r_te, y_r_tr, y_r_te = train_test_split(
    X_all, y_all, test_size=0.2,
    random_state=RANDOM_STATE, stratify=y_all)
df_r_tr = pd.concat([X_r_tr, y_r_tr], axis=1)
df_r_te = pd.concat([X_r_te, y_r_te], axis=1)

result = preprocess_multiclass(df_r_tr, df_r_te)
assert result[0] is not None, "Random multiclass split failed"
(X_tr_rand_mc, y_tr_rand_mc, X_te_rand_mc, y_te_rand_mc,
 le_rand_mc, feats_rand_mc, unseen_rand, n_unseen_rand,
 n_total_te_rand, n_classes_rand, n_known_test_rand) = result
del df_all, df_r_tr, df_r_te, X_all, y_all; gc.collect()
print('Random multiclass split ready ✓')


def make_models(n_classes):
    return {
        'RandomForest': RandomForestClassifier(
            n_estimators=100, max_depth=10, class_weight='balanced',
            random_state=RANDOM_STATE, n_jobs=-1),
        'LightGBM': lgb.LGBMClassifier(
            n_estimators=100, max_depth=8, num_leaves=31,
            subsample=0.8, colsample_bytree=0.8, class_weight='balanced',
            objective='multiclass', num_class=n_classes,
            random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
        'XGBoost': xgb.XGBClassifier(
            n_estimators=100, max_depth=6, subsample=0.8, colsample_bytree=0.8,
            objective='multi:softprob', num_class=n_classes,
            eval_metric='mlogloss',
            random_state=RANDOM_STATE, n_jobs=-1, verbosity=0),
    }

results_mc = []
print('\n' + '='*70)
print('MULTICLASS TEMPORAL LEAKAGE AUDIT (FINAL, CORRECTED)')
print('='*70)

print(f'\n[Random split: {n_classes_rand} classes, {n_known_test_rand} present in test]')
models_rand = make_models(n_classes_rand)
for model_name, model in models_rand.items():
    print(f'--- {model_name} ---')
    model.fit(X_tr_rand_mc, y_tr_rand_mc)
    results_mc.append(evaluate_multiclass(
        model, X_te_rand_mc, y_te_rand_mc, le_rand_mc,
        model_name, 'Random', unseen_rand, n_unseen_rand,
        n_total_te_rand, n_known_test_rand))
    gc.collect()

print(f'\n[Temporal split: {n_classes_temp} classes, {n_known_test_temp} present in test]')
models_temp = make_models(n_classes_temp)
for model_name, model in models_temp.items():
    print(f'--- {model_name} ---')
    model.fit(X_tr_temp_mc, y_tr_temp_mc)
    results_mc.append(evaluate_multiclass(
        model, X_te_temp_mc, y_te_temp_mc, le_temp_mc,
        model_name, 'Temporal', unseen_temp, n_unseen_temp,
        n_total_te_temp, n_known_test_temp))
    gc.collect()

df_mc = pd.DataFrame(results_mc)
print('\n\n' + '='*78)
print('TABLE — MULTICLASS TEMPORAL LEAKAGE AUDIT (FINAL)')
print('='*78)
print(df_mc.to_string(index=False))

print(f'\nSTRUCTURAL FINDING — Friday unseen-class fraction (dataset property):')
print(f'  {unseen_temp} → {n_unseen_temp:,}/{n_total_te_temp:,} rows '
      f'= {n_unseen_temp/n_total_te_temp*100:.1f}% of Friday traffic')

df_mc.to_csv(SAVE_PATH + '/multiclass_inflation_audit_FINAL.csv', index=False)
print(f'\nSaved → multiclass_inflation_audit_FINAL.csv')

monday      :  529,481 rows
tuesday     :  445,645 rows
wednesday   :  691,406 rows
thursday    :  458,626 rows
friday      :  702,718 rows
Total: 2,827,876

Building TEMPORAL multiclass split...
  Features: 43 (port columns excluded) | Train classes: 10
  Train dist: {'BENIGN': 174799, 'DoS GoldenEye': 968, 'DoS Hulk': 21657, 'DoS Slowhttptest': 517, 'DoS slowloris': 545, 'FTP-Patator': 746, 'Heartbleed': 6, 'SSH-Patator': 554, 'Web Attack - Brute Force': 141, 'Web Attack - XSS': 61}
  Unseen-in-test classes: ['Bot', 'DDoS', 'PortScan'] (41,094 rows)
  Known classes actually present in test: 1
  After SMOTE: 356,456 | Test (known): 58,904 | Test (unseen): 41,094
Temporal multiclass split ready ✓

Building RANDOM multiclass split...
  Features: 43 (port columns excluded) | Train classes: 12
  Train dist: {'BENIGN': 128512, 'DoS Hulk': 13020, 'DDoS': 7243, 'PortScan': 8985, 'DoS Slowhttptest': 310, 'Bot': 110, 'DoS slowloris': 327, 'DoS GoldenEye': 582, 'SSH-Patator': 334, 'FTP-Patator'

## Verification: Borderline Class Stress TestChecks every class (not just the ones already known to be problematic) for zero-variance features that could indicate an unresolved data quality issue.

In [3]:
import pandas as pd
import numpy as np
from collections import Counter

print("="*70)
print("BORDERLINE CLASS STRESS TEST — checking all classes for")
print("zero/near-zero variance features that could repeat the")
print("Destination Port pathology under a different feature")
print("="*70)

# class name lookup
class_names = {i: cls for i, cls in enumerate(le_rand_mc.classes_)}

# pre-SMOTE counts for context
pre_smote_counts = {}
df_check = pd.concat(
    [day_dfs[d] for d in ordered_days], ignore_index=True
)
df_check = df_check[~df_check[LABEL_COL].isin(ULTRA_RARE_EXCLUDE)]
print("\nReal (pre-SMOTE, full dataset) sample counts per class:")
print(df_check[LABEL_COL].value_counts())

print("\n" + "-"*70)
print("Per-class feature variance check on POST-SMOTE training data")
print("(low std relative to BENIGN baseline = potential risk)")
print("-"*70)

benign_std = X_tr_rand_mc[(y_tr_rand_mc == 0).values].std()

risk_summary = []
for cls_idx in sorted(y_tr_rand_mc.unique()):
    if cls_idx == 0:
        continue  # skip BENIGN baseline itself
    mask = (y_tr_rand_mc == cls_idx).values
    X_cls = X_tr_rand_mc[mask]
    cls_std = X_cls.std()

    # ratio of this class's std to BENIGN's std, per feature
    # near-zero ratio = near-zero variance relative to baseline
    ratio = (cls_std / benign_std.replace(0, np.nan)).fillna(0)
    n_near_zero = (cls_std < 1e-6).sum()
    n_low_ratio  = (ratio < 0.01).sum()  # std less than 1% of BENIGN's std

    dup_frac = 1 - X_cls.drop_duplicates().shape[0] / len(X_cls)

    real_count = pre_smote_counts.get(cls_idx,
        Counter(y_tr_rand_mc)[cls_idx])  # post-SMOTE count (always 20000 or maj)

    print(f"\nClass {cls_idx} ({class_names[cls_idx]}):")
    print(f"  Post-SMOTE rows: {mask.sum()}")
    print(f"  Near-zero-std features (std<1e-6): {n_near_zero} / {len(cls_std)}")
    print(f"  Low-variance-ratio features (std<1% of BENIGN): {n_low_ratio} / {len(cls_std)}")
    print(f"  Duplicate row fraction: {dup_frac:.4f}")

    if n_near_zero > 0:
        flagged_feats = cls_std[cls_std < 1e-6].index.tolist()
        print(f"  >>> FLAGGED zero-variance features: {flagged_feats}")

    risk_summary.append({
        'class_idx': cls_idx,
        'class_name': class_names[cls_idx],
        'near_zero_std_features': n_near_zero,
        'low_ratio_features': n_low_ratio,
        'duplicate_fraction': round(dup_frac, 4),
    })

print("\n" + "="*70)
print("SUMMARY TABLE — sorted by risk (near-zero-std feature count)")
print("="*70)
risk_df = pd.DataFrame(risk_summary).sort_values(
    'near_zero_std_features', ascending=False)
print(risk_df.to_string(index=False))

print("\nReal (pre-SMOTE) sample counts for the classes flagged above:")
for _, row in risk_df[risk_df['near_zero_std_features'] > 0].iterrows():
    real_n = df_check[df_check[LABEL_COL] == row['class_name']].shape[0]
    print(f"  {row['class_name']}: {real_n} real samples in full dataset")

BORDERLINE CLASS STRESS TEST — checking all classes for
zero/near-zero variance features that could repeat the
Destination Port pathology under a different feature

Real (pre-SMOTE, full dataset) sample counts per class:
Label
BENIGN                      2271320
DoS Hulk                     230124
PortScan                     158804
DDoS                         128025
DoS GoldenEye                 10293
FTP-Patator                    7935
SSH-Patator                    5897
DoS slowloris                  5796
DoS Slowhttptest               5499
Bot                            1956
Web Attack - Brute Force       1507
Web Attack - XSS                652
Heartbleed                       11
Name: count, dtype: int64

----------------------------------------------------------------------
Per-class feature variance check on POST-SMOTE training data
(low std relative to BENIGN baseline = potential risk)
----------------------------------------------------------------------

Class 1 (Bot):
  Po

## Verification: Real vs. Synthetic Zero-Variance FeaturesConfirms the zero-variance features identified above are genuine attack signatures present in the raw, pre-SMOTE data — not an artefact of synthetic oversampling.

In [4]:
import pandas as pd
import numpy as np

print("="*70)
print("VERIFYING: are these zero-variance features REAL attack behavior,")
print("or introduced/amplified by SMOTE?")
print("="*70)

# rebuild real (pre-SMOTE) feature data for the flagged classes,
# using the SAME scaler that was fit during preprocess_multiclass
# so the comparison is apples-to-apples with the post-SMOTE numbers

flagged_features = ['FIN Flag Count', 'Fwd PSH Flags', 'URG Flag Count',
                     'Active Mean', 'Active Std', 'Active Max', 'Active Min',
                     'Idle Std', 'Bwd Packet Length Min', 'Fwd Packet Length Min',
                     'Min Packet Length', 'Fwd Header Length']

# use the RAW (unscaled) data straight from day_dfs for direct interpretability
df_real = pd.concat(
    [day_dfs[d] for d in ordered_days], ignore_index=True
)
df_real = df_real[~df_real[LABEL_COL].isin(ULTRA_RARE_EXCLUDE)]

available_flagged = [f for f in flagged_features if f in df_real.columns]
print(f"\nChecking {len(available_flagged)} flagged features on RAW real data "
      f"(no scaling, no SMOTE):\n")

for cls in df_real[LABEL_COL].unique():
    sub = df_real[df_real[LABEL_COL] == cls]
    n_real = len(sub)
    zero_var_real = []
    for feat in available_flagged:
        if sub[feat].std() < 1e-9:
            zero_var_real.append((feat, sub[feat].iloc[0]))
    if zero_var_real:
        print(f"{cls} (n={n_real:,}):")
        for feat, val in zero_var_real:
            print(f"    {feat} = constant at {val} across ALL {n_real:,} real samples")
        print()

print("="*70)
print("Interpretation: if a feature is constant across thousands or")
print("hundreds of thousands of REAL samples (not synthetic), it is a")
print("genuine attack signature, not a SMOTE or preprocessing artifact.")
print("="*70)

VERIFYING: are these zero-variance features REAL attack behavior,
or introduced/amplified by SMOTE?

Checking 12 flagged features on RAW real data (no scaling, no SMOTE):

FTP-Patator (n=7,935):
    FIN Flag Count = constant at 0 across ALL 7,935 real samples
    Active Mean = constant at 0.0 across ALL 7,935 real samples
    Active Std = constant at 0.0 across ALL 7,935 real samples
    Active Max = constant at 0 across ALL 7,935 real samples
    Active Min = constant at 0 across ALL 7,935 real samples
    Idle Std = constant at 0.0 across ALL 7,935 real samples
    Bwd Packet Length Min = constant at 0 across ALL 7,935 real samples
    Min Packet Length = constant at 0 across ALL 7,935 real samples

SSH-Patator (n=5,897):
    FIN Flag Count = constant at 0 across ALL 5,897 real samples
    Active Mean = constant at 0.0 across ALL 5,897 real samples
    Active Std = constant at 0.0 across ALL 5,897 real samples
    Active Max = constant at 0 across ALL 5,897 real samples
    Active Mi

## Verification: VarianceThreshold Interaction CheckConfirms these features correctly survive the VarianceThreshold(0.01) feature-selection step and are present in the final feature set used for model training.

In [5]:
import pandas as pd
import numpy as np

print("="*70)
print("VARIANCE THRESHOLD INTERACTION CHECK")
print("Confirming these per-class-constant features survive the")
print("VarianceThreshold(0.01) step because they have nonzero")
print("variance OVERALL (across all classes combined), even though")
print("they're constant WITHIN each individual class")
print("="*70)

flagged_features = ['FIN Flag Count', 'Fwd PSH Flags', 'URG Flag Count',
                     'Active Mean', 'Active Std', 'Active Max', 'Active Min',
                     'Idle Std', 'Bwd Packet Length Min', 'Fwd Packet Length Min',
                     'Min Packet Length', 'Fwd Header Length']

df_real = pd.concat(
    [day_dfs[d] for d in ordered_days], ignore_index=True
)
df_real = df_real[~df_real[LABEL_COL].isin(ULTRA_RARE_EXCLUDE)]

available_flagged = [f for f in flagged_features if f in df_real.columns]

print(f"\nOverall (cross-class) variance for each flagged feature,")
print(f"computed on the SAME multiclass training population used")
print(f"in the Random-split audit:\n")

# use the random-split training population specifically, since that's
# what VarianceThreshold(0.01) was actually fit on in preprocess_multiclass
for feat in available_flagged:
    if feat not in df_real.columns:
        continue
    overall_var = df_real[feat].var()
    overall_std = df_real[feat].std()
    n_unique_vals = df_real[feat].nunique()

    # per-class breakdown: which classes have this at 0 vs nonzero
    by_class = df_real.groupby(LABEL_COL)[feat].agg(['mean','std','nunique'])
    n_classes_zero    = (by_class['std'].fillna(0) < 1e-9).sum()
    n_classes_nonzero = len(by_class) - n_classes_zero

    print(f"{feat}:")
    print(f"  Overall variance: {overall_var:.6f} (threshold cutoff: 0.01)")
    print(f"  Overall std: {overall_std:.6f} | unique values: {n_unique_vals}")
    print(f"  Classes where this feature is constant: {n_classes_zero}")
    print(f"  Classes where this feature varies: {n_classes_nonzero}")
    print(f"  → {'SURVIVES' if overall_var >= 0.01 else 'WOULD BE DROPPED BY'} VarianceThreshold(0.01)")
    print()

print("="*70)
print("Note: VarianceThreshold operates on the SCALED (standardized)")
print("feature matrix in the actual pipeline, not raw values shown above.")
print("After StandardScaler, every retained feature has variance ≈1.0")
print("by construction (overall), so VarianceThreshold(0.01) essentially")
print("never drops anything post-scaling unless a feature was already")
print("fully constant pre-scaling (std=0 → scaler can't standardize it,")
print("typically yields 0 or NaN, also caught separately).")
print("="*70)

# direct confirmation: check if any flagged feature was ACTUALLY
# dropped during the real Random-split preprocessing run
print(f"\nDirect check — are these features present in the FINAL")
print(f"feature set actually used in the Random-split model (feats_rand_mc)?\n")
for feat in available_flagged:
    present = feat in feats_rand_mc
    print(f"  {feat}: {'PRESENT in final 43 features' if present else 'DROPPED before model training'}")

VARIANCE THRESHOLD INTERACTION CHECK
Confirming these per-class-constant features survive the
VarianceThreshold(0.01) step because they have nonzero
variance OVERALL (across all classes combined), even though
they're constant WITHIN each individual class

Overall (cross-class) variance for each flagged feature,
computed on the SAME multiclass training population used
in the Random-split audit:

FIN Flag Count:
  Overall variance: 0.034038 (threshold cutoff: 0.01)
  Overall std: 0.184495 | unique values: 2
  Classes where this feature is constant: 10
  Classes where this feature varies: 3
  → SURVIVES VarianceThreshold(0.01)

Fwd PSH Flags:
  Overall variance: 0.044238 (threshold cutoff: 0.01)
  Overall std: 0.210328 | unique values: 2
  Classes where this feature is constant: 8
  Classes where this feature varies: 5
  → SURVIVES VarianceThreshold(0.01)

URG Flag Count:
  Overall variance: 0.085836 (threshold cutoff: 0.01)
  Overall std: 0.292977 | unique values: 2
  Classes where this 